In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.datasets.loader import (
    DATASET_IDS,
    load_documents,
    load_queries,
    load_qrels,
)

In [7]:
DATASET_IDS

{'scifact': 'beir/scifact/test', 'nfcorpus': 'beir/nfcorpus/test'}

In [ ]:
docs = load_documents("main")
queries = load_queries("main")
qrels = load_qrels("main")

[INFO] [starting] building docstore
[INFO] [starting] opening zip file                                              
[INFO] [starting] https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip
docs_iter:   0%|                                      | 0/3633 [00:01<?, ?doc/s]
[INFO] [finished] https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip: [00:14] [2.45MB] [169kB/s]
[INFO] [finished] opening zip file [15.58s]                                     
docs_iter: 100%|██████████████████████████| 3633/3633 [00:15<00:00, 229.34doc/s]
[INFO] [finished] docs_iter: [00:15] [3633doc] [229.32doc/s]
[INFO] [finished] building docstore [15.85s]
[INFO] [starting] opening zip file
[INFO] [finished] opening zip file s]
[INFO] [starting] opening zip file
[INFO] [finished] opening zip file s]


In [9]:
print("SciFact documents:")
display(docs_scifact.head())

print("NFCorpus documents:")
display(docs_nfcorpus.head())

SciFact documents:


,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...


NFCorpus documents:


,doc_id,text,title,url
0,MED-10,"Recent studies have suggested that statins, an...",Statin Use and Breast Cancer Survival: A Natio...,http://www.ncbi.nlm.nih.gov/pubmed/25329299
1,MED-14,BACKGROUND: Preclinical studies have shown tha...,Statin use after diagnosis of breast cancer an...,http://www.ncbi.nlm.nih.gov/pubmed/25304447
2,MED-118,The aims of this study were to determine the c...,Alkylphenols in human milk and their relations...,http://www.ncbi.nlm.nih.gov/pubmed/20435081%20
3,MED-301,Epilepsy or seizure disorder is one of the mos...,Methylmercury: A Potential Environmental Risk ...,http://www.ncbi.nlm.nih.gov/pubmed/22206970
4,MED-306,Hit Reaction Time latencies (HRT) in the Conti...,Sensitivity of Continuous Performance Test (CP...,http://www.ncbi.nlm.nih.gov/pubmed/20699117


In [10]:
print("SciFact columns:", docs_scifact.columns.tolist())
print("NFCorpus columns:", docs_nfcorpus.columns.tolist())

SciFact columns: ['doc_id', 'text', 'title']
NFCorpus columns: ['doc_id', 'text', 'title', 'url']


In [11]:
print("SciFact queries:")
display(queries_scifact.head())

print("SciFact qrels:")
display(qrels_scifact.head())

print("NFCorpus queries:")
display(queries_nfcorpus.head())

print("NFCorpus qrels:")
display(qrels_nfcorpus.head())

SciFact queries:


,query_id,text
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...


SciFact qrels:


,query_id,doc_id,relevance,iteration
0,1,31715818,1,0
1,3,14717500,1,0
2,5,13734012,1,0
3,13,1606628,1,0
4,36,5152028,1,0


NFCorpus queries:


,query_id,text,url
0,PLAIN-2,Do Cholesterol Statin Drugs Cause Breast Cancer?,http://nutritionfacts.org/2015/07/16/do-choles...
1,PLAIN-12,Exploiting Autophagy to Live Longer,http://nutritionfacts.org/2015/06/11/exploitin...
2,PLAIN-23,How to Reduce Exposure to Alkylphenols Through...,http://nutritionfacts.org/2015/04/28/how-to-re...
3,PLAIN-33,What’s Driving America’s Obesity Problem?,http://nutritionfacts.org/2015/03/24/whats-dri...
4,PLAIN-44,Who Should be Careful About Curcumin?,http://nutritionfacts.org/2015/02/12/who-shoul...


NFCorpus qrels:


,query_id,doc_id,relevance,iteration
0,PLAIN-2,MED-2427,2,0
1,PLAIN-2,MED-10,2,0
2,PLAIN-2,MED-2429,2,0
3,PLAIN-2,MED-2430,2,0
4,PLAIN-2,MED-2431,2,0


In [ ]:
import pandas as pd


def build_text_column(documents: pd.DataFrame) -> pd.Series:
    """
    Merge title and text fields safely for inspection.
    """
    text_parts = []

    for column in ["title", "text", "abstract"]:
        if column in documents.columns:
            text_parts.append(documents[column].fillna("").astype(str))

    if not text_parts:
        raise ValueError("No textual document fields were found.")

    merged_text = text_parts[0]

    for part in text_parts[1:]:
        merged_text = merged_text + " " + part

    return merged_text.str.strip()


def summarize_dataset(
    name: str,
    documents: pd.DataFrame,
    queries: pd.DataFrame,
    qrels: pd.DataFrame,
) -> dict:
    text = build_text_column(documents)
    token_counts = text.str.split().str.len()

    return {
        "dataset": name,
        "documents_count": len(documents),
        "queries_count": len(queries),
        "qrels_count": len(qrels),
        "unique_documents": documents["doc_id"].nunique(),
        "unique_queries": queries["query_id"].nunique(),
        "duplicate_doc_ids": int(documents["doc_id"].duplicated().sum()),
        "empty_documents": int(text.eq("").sum()),
        "average_document_words": round(float(token_counts.mean()), 2),
        "median_document_words": round(float(token_counts.median()), 2),
        "min_relevance": int(qrels["relevance"].min()),
        "max_relevance": int(qrels["relevance"].max()),
    }

audit_df = pd.DataFrame([
    summarize_dataset(
        "main",
        docs,
        queries,
        qrels,
    )
])

display(audit_df)

,dataset,documents_count,queries_count,qrels_count,unique_documents,unique_queries,duplicate_doc_ids,empty_documents,average_document_words,median_document_words,min_relevance,max_relevance
0,beir/scifact/test,5183,300,339,5183,300,0,0,214.63,204.0,1,1
1,beir/nfcorpus/test,3633,323,12334,3633,323,0,0,233.77,237.0,1,2


In [14]:
print("SciFact relevance distribution:")
display(
    qrels_scifact["relevance"]
    .value_counts()
    .sort_index()
    .rename_axis("relevance")
    .reset_index(name="count")
)

print("NFCorpus relevance distribution:")
display(
    qrels_nfcorpus["relevance"]
    .value_counts()
    .sort_index()
    .rename_axis("relevance")
    .reset_index(name="count")
)

SciFact relevance distribution:


,relevance,count
0,1,339


NFCorpus relevance distribution:


,relevance,count
0,1,11758
1,2,576


In [15]:
from pathlib import Path

output_dir = PROJECT_ROOT / "reports"
output_dir.mkdir(parents=True, exist_ok=True)

audit_df.to_csv(
    output_dir / "dataset_audit.csv",
    index=False,
)

print("Saved:", output_dir / "dataset_audit.csv")

Saved: e:\in Desktop\IR\reports\dataset_audit.csv
